# Section 1

## Import Necessary Libraries

In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

print(torch.__version__)
print(torch.cuda.is_available())

2.11.0+cu130
True


## Download data

In [2]:
# Download data from open datasets.
training_data = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor(),
)

# Download test data from open datasets.
test_data = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=ToTensor(),
)

## Batch-ing

In [3]:
batch_size = 64

# Create data loaders.
train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

for X, y in test_dataloader:
    print(f"Shape of X [N, C, H, W]: {X.shape}")
    print(f"Shape of y: {y.shape} {y.dtype}")
    break

Shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
Shape of y: torch.Size([64]) torch.int64


## Models


In [4]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

# Define model
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

model = NeuralNetwork().to(device)
print(model)

Using cuda device
NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (linear_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


## Optimizing Model Parameters

In [5]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)

## Train Function

In [6]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)

        # Backpropagation
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()

        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

## Test Function

In [7]:
def test(dataloader, model, loss_fn):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

## Run the train-test

In [8]:
epochs = 20
for t in range(epochs):
    print(f"Epoch {t+1}\n-------------------------------")
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn)
print("Done!")

Epoch 1
-------------------------------
loss: 2.304299  [   64/60000]
loss: 2.282641  [ 6464/60000]
loss: 2.268193  [12864/60000]
loss: 2.262617  [19264/60000]
loss: 2.240741  [25664/60000]
loss: 2.213441  [32064/60000]
loss: 2.217808  [38464/60000]
loss: 2.183941  [44864/60000]
loss: 2.188863  [51264/60000]
loss: 2.154541  [57664/60000]
Test Error: 
 Accuracy: 55.0%, Avg loss: 2.143345 

Epoch 2
-------------------------------
loss: 2.151155  [   64/60000]
loss: 2.134682  [ 6464/60000]
loss: 2.078451  [12864/60000]
loss: 2.107437  [19264/60000]
loss: 2.042035  [25664/60000]
loss: 1.977901  [32064/60000]
loss: 2.015049  [38464/60000]
loss: 1.932142  [44864/60000]
loss: 1.946990  [51264/60000]
loss: 1.872372  [57664/60000]
Test Error: 
 Accuracy: 58.3%, Avg loss: 1.862304 

Epoch 3
-------------------------------
loss: 1.892032  [   64/60000]
loss: 1.854003  [ 6464/60000]
loss: 1.739405  [12864/60000]
loss: 1.795940  [19264/60000]
loss: 1.672945  [25664/60000]
loss: 1.619179  [32064/600

## Save the Model

In [9]:
torch.save(model.state_dict(), "model/model.pth")
print("Saved PyTorch Model State to model.pth")

Saved PyTorch Model State to model.pth


## Load Models

In [10]:
model = NeuralNetwork().to(device)
model.load_state_dict(torch.load("model/model.pth", weights_only=True))

<All keys matched successfully>

In [11]:
classes = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot",
]

model.eval()
x, y = test_data[0][0], test_data[0][1]
with torch.no_grad():
    x = x.to(device)
    pred = model(x)
    predicted, actual = classes[pred[0].argmax(0)], classes[y]
    print(f'Predicted: "{predicted}", Actual: "{actual}"')

Predicted: "Ankle boot", Actual: "Ankle boot"


# Tensors

## Import Necessary Libraries

In [12]:
import torch 
import numpy as np

## Tensor Testing

In [13]:
data = [[1, 2],[3, 4]]
x_data = torch.tensor(data)

print(x_data)
print(x_data.dtype)

tensor([[1, 2],
        [3, 4]])
torch.int64


In [14]:
np_array = np.array(data)   
x_np = torch.from_numpy(np_array)

print(x_np)
print(x_np.dtype)

tensor([[1, 2],
        [3, 4]])
torch.int64


In [15]:
x_ones = torch.ones_like(x_data) # retains the properties of x_data
print(f"Ones Tensor: \n {x_ones} \n")

x_rand = torch.rand_like(x_data, dtype=torch.float) # overrides the datatype of x_data
print(f"Random Tensor: \n {x_rand} \n")

Ones Tensor: 
 tensor([[1, 1],
        [1, 1]]) 

Random Tensor: 
 tensor([[0.1604, 0.8273],
        [0.5263, 0.9431]]) 



In [16]:
shape = (2,3,)
rand_tensor = torch.rand(shape)
ones_tensor = torch.ones(shape)
zeros_tensor = torch.zeros(shape)

print(f"Random Tensor: \n {rand_tensor} \n")
print(f"Ones Tensor: \n {ones_tensor} \n")
print(f"Zeros Tensor: \n {zeros_tensor}")

Random Tensor: 
 tensor([[0.0290, 0.7607, 0.1161],
        [0.8866, 0.8004, 0.0110]]) 

Ones Tensor: 
 tensor([[1., 1., 1.],
        [1., 1., 1.]]) 

Zeros Tensor: 
 tensor([[0., 0., 0.],
        [0., 0., 0.]])


## Tensor Attributes

In [17]:
tensor = torch.rand(3,4)

print(f"Data: {tensor}")
print(f"Shape of tensor: {tensor.shape}")
print(f"Datatype of tensor: {tensor.dtype}")
print(f"Device tensor is stored on: {tensor.device}")

Data: tensor([[0.3447, 0.7889, 0.4949, 0.9200],
        [0.0307, 0.0905, 0.5814, 0.6157],
        [0.2347, 0.8442, 0.9369, 0.5778]])
Shape of tensor: torch.Size([3, 4])
Datatype of tensor: torch.float32
Device tensor is stored on: cpu


## Tensor Operations

In [18]:
# We move our tensor to the current accelerator if available
if torch.accelerator.is_available():
    print(torch.accelerator.current_accelerator())
    tensor = tensor.to(torch.accelerator.current_accelerator())

cuda


In [19]:
tensor = torch.rand(4, 4)
print(f"First row: {tensor[0]}")
print(f"First column: {tensor[:, 0]}")
print(f"Last column: {tensor[..., -1]}")
tensor[:,1] = 0
print(tensor)

First row: tensor([0.9802, 0.5960, 0.1168, 0.8335])
First column: tensor([0.9802, 0.0032, 0.0294, 0.0430])
Last column: tensor([0.8335, 0.5653, 0.2487, 0.9470])
tensor([[0.9802, 0.0000, 0.1168, 0.8335],
        [0.0032, 0.0000, 0.6448, 0.5653],
        [0.0294, 0.0000, 0.5237, 0.2487],
        [0.0430, 0.0000, 0.3369, 0.9470]])
